# Chapter 5 · Brownian Bridges for ABPs — Inference with PPO-learned Bridges

Section 5.8 (pp. 107–124), Figures 45–51.

| Figure | Page | Content |
|---|---|---|
| Figure 43 | p. 108 | Correlation of system parameters and bridges (illustration, not computed) |
| Figure 44 | p. 115 | Reinforcement-learning step (illustration, not computed) |
| Figure 45 | p. 117 | Likelihood and posterior from particle filters |
| Figure 46 | p. 119 | Bridge-parameter posterior |
| Figure 47 | p. 120 | PPO training progress |
| Figure 48 | p. 121 | Posterior approximation by PPO-learned bridges |
| Figure 49 | p. 123 | Bootstrapped MSD fit |
| Figure 50 | p. 123 | Accuracy of different velocity estimators |
| Figure 51 | p. 124 | Inference in a harmonic potential |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

import lambert_thesis as lt
from lambert_thesis import bridge_learning as bl, bridges, plotting
from lambert_thesis.plotting import savefig

plotting.setup()
rng = lt.set_seed(lt.SEED)

## 5.8 Inference

We now infer the velocity $v_0$ of a Ballistic Brownian particle. When proposing new system parameters, the optimal bridges
change with them (Figure 43): a slow particle can reach an observation on a straight path, a fast one needs to wiggle. We
therefore construct an offset function $f_o(v_0)$ of the bridge shift and slope that yields the highest posterior density
for a given $v_0$, and shear the auxiliary space with it — a volume-preserving operation that does not alter the posterior.

### Constructing the offset function with Proximal Policy Optimization
$f_o$ is a fully connected network with hidden layers of 16 units, trained with PPO (CleanRL implementation). The policy
$\pi_\theta=\mathcal N(\mathbf f_o,\sigma_c)$ produces the bridge for the next step from the next $f=3$ observations,
expressed relative to the swarm's centre of mass and rotated into its mean orientation, plus the proposed velocity. The
reward of a step is the shifted observation log-likelihood,

$$R_k=\begin{cases}\log p(y_k\mid v_0,b_1^k,b_2^k)+r_0, & \log p(y_k\mid v_0,b_1^k,b_2^k) > -r_0\\ 0 & \text{otherwise (episode truncated)},\end{cases}$$

with discount $\gamma=0.5$. Training: draw $v_0$ from a training range, run the filter while sampling bridges from the policy,
collect 1024 state-action-reward triplets, update the policy, repeat; finally use only the policy mean $f_o$. The observation
precision is annealed (starting relaxed by $0.5/\sigma_m$) once the swarm traverses all observations, and an entropy bonus
encourages exploration until then (Figure 44 illustrates one learning step).

*Code:* `ppo.PPOAgent` (actor/critic networks, GAE, clipped PPO update), `bridge_learning.run_filter` (particle filter whose
bridges are chosen by the agent, with rewards and truncation), `bridge_learning.train_agent`, `bridge_learning.likelihood_scan`.
`bridge_learning.run_experiment(kind, seed)` runs the whole pipeline for one synthetic observation set.

**Parameters** (appendix): $v_0=1$, $\sigma=D_r=0.5$, observations every $\Delta T=1$ ($\delta t=0.1$ inside the filter),
30 observations, measurement noise $\sigma_m=0.3$, 3 future observations as features.

**Switch:** `FULL_RESOLUTION = True` uses 15 trained agents and 50 observation sets each for Figures 50 and 51 (many
hours); `False` uses 3, 3 and 5. Every experiment is stored in `results/` and reused.

In [ ]:
FULL_RESOLUTION = True
N_AGENTS = 15 if FULL_RESOLUTION else 3
N_SETS_FREE = 50 if FULL_RESOLUTION else 3
N_SETS_WELL = 50 if FULL_RESOLUTION else 5
EXAMPLE = 7                                 # observation set of Figures 45-49 (seed 7)
RES = lt.REPO_ROOT / "results"

main = bl.run_experiments("free", [EXAMPLE], RES / "bridges_free")[0]

## Results

For each experiment we run the filter with a reasonable number of particles (4000) and, for validation, with 100 000
particles, probing the likelihood on a grid of velocities and normalising it (flat prior).

### Figure 45 (p. 117)

In [ ]:
norm_lik = lambda logl: np.exp(logl - logl.max())
v, l = main["naive_v"], norm_lik(main["naive_logl"])
vs, ls = main["sr_v"], norm_lik(main["sr_logl"])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.semilogy(v, l, c="orange", lw=1, label="particle filter, 4000 particles")
a1.semilogy(vs, ls, c="red", lw=1, label="particle filter, 100 000 particles")
a1.set(xlabel=r"$v / v_{GT}$", ylabel="Likelihood")
a2.plot(v, l / np.trapezoid(l, v), c="orange", lw=1, label="particle filter, 4000 particles")
a2.plot(vs, ls / np.trapezoid(ls, vs), c="red", lw=1, label="particle filter, 100 000 particles")
a2.set(xlabel=r"$v / v_{GT}$", ylabel="Posterior Probability")
for a in (a1, a2):
    a.axvline(1.0, c="k", label="ground truth"); a.grid(); a.legend(fontsize=8)
fig.tight_layout()
savefig("fig45_particle_filter_likelihood");

**Figure 45 (p. 117).** Likelihood (left) and posterior (right) for the velocity, normalised with the ground truth, from 30
observations: 4000 particles (yellow) and 100 000 particles (red). Fewer particles are cheaper per run but give noisier
estimates, so more runs are needed, e.g. in an MCMC sampling.

### Figure 46 (p. 119): bridge-parameter posterior
The filter also estimates the bridge parameters (final shift and slope of the noise) of every interval. For each observation
we show Gaussian fits (density levels 10 %, 50 %, 90 %) of the bridge parameters along the posterior trajectories over the
velocity grid.

In [ ]:
LEVELS = (0.1, 0.5, 0.9)
fig, axes = plt.subplots(5, 6, figsize=(15, 10))
for k, ax in enumerate(axes.flat):
    for lev in LEVELS:
        e = bridges.gaussian_level_ellipse(bridges.bridge_prior_cov(11), lev)
        ax.fill(*e, color="k", alpha=0.5, lw=0)
    for lev in LEVELS:
        ax.fill(*bridges.empirical_level_ellipse(main["sr_shift"][:, k], main["sr_slope"][:, k], lev), color="red", alpha=0.5, lw=0)
    ax.plot(*bridges.empirical_level_ellipse(main["naive_shift"][:, k], main["naive_slope"][:, k], LEVELS[0]), color="orange")
    for lev in LEVELS:
        ax.fill(*bridges.empirical_level_ellipse(main["bridge_shift"][:, k], main["bridge_slope"][:, k], lev), color="lime", alpha=0.5, lw=0)
    ax.axvline(main["bridge_shift"][:, k].mean(), lw=0.5, c="green")
    ax.axhline(main["bridge_slope"][:, k].mean(), lw=0.5, c="green")
    kl = bridges.kl_divergence_from_samples(11, main["bridge_shift"][:, k], main["bridge_slope"][:, k])
    ax.set_title(f"KL({k})={kl:.1f} bit")
handles = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.5) for c in ("k", "red", "lime")] + [plt.Line2D([], [], color="orange")]
fig.legend(handles, ["prior of the bridge parameters", "posterior, 100 000 particles", "PPO agent (policy mean)",
                     "posterior, 4000 particles (10 % level)"], loc="lower center", ncol=4)
fig.tight_layout(rect=(0, 0.04, 1, 1))
savefig("fig46_bridge_posterior");

**Figure 46 (p. 119).** Slope (y-axis) vs. shift (x-axis) for each of the 30 observations. Grey: prior of the bridge parameters,
which the Euler–Maruyama algorithm naturally samples. Yellow and red: bridge parameters in the posterior after assimilating the
observations (here only observed, not enforced). Title: KL divergence of the bridges of the PPO agent (green) from the prior.
The learned bridges do not always align with the peak of the posterior; some change with $v_0$ (elongated streaks), others not.

### Figure 47 (p. 120): training progress

The snapshots show the last episode of the first and of the last training step.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, step in zip(axes, (0, len(main["train_snapshot_episodes"]) - 1)):
    traj, n_ep = main["train_snapshot_trajectory"][step], int(main["train_snapshot_episodes"][step])
    ax.plot(main["gt_x"], main["gt_y"], "-", alpha=0.5, label="ground truth")
    ax.plot(main["obs_x"], main["obs_y"], "xr", alpha=0.5, label="observations")
    ax.plot(traj[0], traj[1], "-k", lw=2, label="last episode of the agent")
    ax.set(xlim=(main["obs_x"].min() - 5, main["obs_x"].max() + 5), ylim=(main["obs_y"].min() - 5, main["obs_y"].max() + 5),
           title=f"{n_ep} episodes seen")
    ax.legend(fontsize=8)
savefig("fig47ab_training_snapshots");

In [ ]:
plt.figure()
plt.plot(main["train_length"], lw=1, label="Episode_Length")
plt.plot(main["train_reward"], lw=1, label="Episode_Reward")
plt.xlabel("Episode"); plt.ylabel("Value"); plt.grid(); plt.legend()
savefig("fig47c_training_length_reward");

**Figure 47 (p. 120).** Top: ground truth (blue), observations (red crosses) and a snapshot of the PPO agent (black). In the
first episodes the reward is low and the episodes are truncated early. Bottom: episode length (observations reached) and
reward. Thanks to the transformation into the swarm's coordinate system, the knowledge from early observations generalises and
the length quickly reaches the full trajectory; the reward stabilises afterwards.

### Figure 48 (p. 121): posterior approximation by the learned bridges
After training, the agent is used deterministically (policy mean). Left: the typical result. Right: independently trained
agents on the same observations (means and standard deviations of their posterior estimates; red: high-resolution filter).

In [ ]:
lb = norm_lik(main["bridge_logl"])
lr_ = np.exp(-(main["bridge_reward"] - main["bridge_reward"].max()) * np.log(bl.REWARD_RANGE))
plt.figure()
plt.plot(v, l / np.trapezoid(l, v), c="orange", lw=1, label="Naive Particle Filter")
plt.plot(main["bridge_v"], lb / np.trapezoid(lb, main["bridge_v"]), c="blue", lw=2, label="Bridges Density")
plt.plot(main["bridge_v"], lr_ / np.trapezoid(lr_, main["bridge_v"]), c="green", lw=2, label="Bridge Reward Density")
plt.plot(vs, ls / np.trapezoid(ls, vs), "--", c="red", lw=1, label="Super Resolution Particle Filter")
plt.xlabel(r"$v_0/v_{GT}$", fontsize=14); plt.ylabel("$p$", fontsize=14, rotation=0); plt.legend()
savefig("fig48a_bridge_posterior_density");

In [ ]:
agents = bl.run_experiments("free", [EXAMPLE * 1000 + i for i in range(N_AGENTS)], RES / "bridges_free_agents",
                            gt_seed=EXAMPLE, agent_only=True)
plt.figure()
plt.axvline(main["sr_mean"], c="r", label="100 000 particles: mean")
plt.axvline(main["sr_mean"] - main["sr_std"], c="r", ls="--", label="... ± standard deviation")
plt.axvline(main["sr_mean"] + main["sr_std"], c="r", ls="--")
for i, r in enumerate(agents):
    pos = i + 1.1
    plt.plot([r["bridge_mean"] - r["bridge_std"], r["bridge_mean"] + r["bridge_std"]], [pos, pos], "b-", lw=2,
             label="PPO agent: mean ± std (thin), CI of the mean (thick)" if i == 0 else None)
    plt.plot(r["bridge_ci_mean"], [pos, pos], "-b", lw=8)
plt.axvline(1.0, c="k", lw=2, label="ground truth")
plt.xlabel(r"$v_0/v_{GT}$"); plt.ylabel("Agent ID"); plt.xlim(0.9, 1.1); plt.legend(fontsize=7)
savefig("fig48b_independent_agents");

**Figure 48 (p. 121).** Left: bridged filter and naive filter with 4000 particles each, the super-resolution filter with
100 000 particles as a reference. The bridge estimate is aligned with the true posterior but underestimates its width.
Right: independent training results on the same observations fluctuate within the true posterior — PPO training is stochastic
(random initialisation, randomised minibatches).

### Figure 49 (p. 123): bootstrapped MSD fit
The standard alternative is a fit of the mean-squared displacement on the log-transformed displacements with bootstrapped
confidence bounds, considering lags up to the rotational diffusion time.

In [ ]:
free_sets = bl.run_experiments("random", [100 + i for i in range(N_SETS_FREE)], RES / "bridges_random")
ex = free_sets[0]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
a1.plot(ex["gt_x"], ex["gt_y"], ".-", alpha=0.5, label="true trajectory")
a1.plot(ex["obs_x"], ex["obs_y"], "xr", alpha=0.5, label="observations")
a1.plot([0], [0], "+b"); a1.axis("equal"); a1.legend(fontsize=8)
for t in ex["msd_curves"]:
    a2.loglog(ex["msd_lags"], t, "k", alpha=0.05)
lag = ex["msd_lags"]
for v0 in ex["msd_v"]:
    a2.plot(lag, bl.abp_msd(lag, v0, bl.GT_DR, bl.SIG_OBS), "r", alpha=0.05)
a2.plot([], [], "k", label="bootstrapped MSD of the observations"); a2.plot([], [], "r", label="fitted theoretical MSD")
a2.set(xlabel="Lag time", ylabel="MSD"); a2.legend(fontsize=8)
savefig("fig49_bootstrap_msd");

**Figure 49 (p. 123).** Left: simulated trajectory (blue) and observations (red crosses). Right: bootstrapped MSD (resampled
displacements) and the theoretical MSD fitted to each sample; only the first two lags (below the rotational diffusion time)
enter the fit. For the largest lags the bootstrap fails, as only one displacement remains.

### Figure 50 (p. 123): accuracy of the estimators

In [ ]:
m_msd = np.array([r["msd_mean"] for r in free_sets]); s_msd = np.array([r["msd_std"] for r in free_sets])
m_pf = np.array([r["naive_mean"] for r in free_sets]); s_pf = np.array([r["naive_std"] for r in free_sets])
m_b = np.array([r["bridge_mean"] for r in free_sets]); s_b = np.array([r["bridge_std"] for r in free_sets])

def estimator_histograms(data, stds, labels, xlim, bins=11):
    plt.figure()
    for d, c in zip(data, "grb"):
        plt.hist(d, color=c, alpha=0.25, bins=bins, density=True)
    x = np.linspace(*xlim, 1000)
    for d, s, c, lab in zip(data, stds, "grb", labels):
        mu, sd = norm.fit(d)
        plt.plot(x, norm.pdf(x, mu, sd), color=c, lw=2, label=f"{lab}: spread of the estimates")
        plt.plot(x, norm.pdf(x, mu, np.mean(s)), color=c, lw=1, ls="--")
    plt.plot([], [], "k--", lw=1, label="reported accuracy (dashed)")
    plt.axvline(1, c="k", lw=2)
    plt.xlabel(r"$v_0/v_{GT}$"); plt.ylabel("$p(v_0)$"); plt.legend(fontsize=8)

estimator_histograms([m_msd, m_pf, m_b], [s_msd, s_pf, s_b], ["MSD", "Particle Filter", "PPO Agent"], (0.9, 1.1))
savefig("fig50a_estimator_accuracy");

In [ ]:
plt.figure()
plt.plot(s_pf, s_b, "ob", ms=4, label="bridged particle filter")
plt.plot(s_pf, s_msd, "xg", ms=4, label="MSD fit")
plt.plot([0, s_pf.max()], [0, s_pf.max()], "--r", label="equal width")
if len(s_pf) > 1:
    k, d = np.polyfit(s_pf, s_b, 1)
    x_ = np.linspace(0, s_pf.max())
    plt.plot(x_, k * x_ + d, "--b", label="linear fit (bridged)")
plt.xlim(0, None); plt.ylim(0, None)
plt.xlabel("Posterior Standard Distribution (particle filter)"); plt.ylabel("Standard Distribution (bridge estimation)")
plt.legend(fontsize=8)
savefig("fig50b_reported_widths");

**Figure 50 (p. 123).** Left: histograms of the estimates from independent observation sets for the bootstrapped MSD fit, the
regular particle filter and the bridged filter; dashed: the accuracy each estimator reports ($1\sigma$). All are unbiased with
comparable accuracy; the bridged filter is overconfident. Right: reported accuracy of MSD (green) and bridged filter (blue)
vs. the posterior width of the standard particle filter.

### Figure 51 (p. 124): harmonic potential
The particle now moves in a harmonic potential centred at the origin (prefactor $k=0.5$). The MSD of free motion cannot
capture the caging, while the filter only needs the equations of motion. Standard filter: 2000 particles; bridged filter: 10.

In [ ]:
well = bl.run_experiments("well", [200 + i for i in range(N_SETS_WELL)], RES / "bridges_well")
w_pf = np.array([r["naive_mean"] for r in well]); ws_pf = np.array([r["naive_std"] for r in well])
w_b = np.array([r["bridge_mean"] for r in well]); ws_b = np.array([r["bridge_std"] for r in well])
w_msd = np.array([r["msd_mean"] for r in well]); ws_msd = np.array([r["msd_std"] for r in well])
ids = np.arange(1, len(well) + 1)

plt.figure()
plt.axvline(0, c="r", label="Posterior")
plt.plot(ws_pf, ids, "-xr"); plt.plot(-ws_pf, ids, "-xr")
plt.plot(1 - w_pf, ids, "-k", lw=2, label="Ground Truth")
plt.plot(w_msd - w_pf, ids, "g", lw=1, label="MSD Estimation")
for i, (mb, sb_) in enumerate(zip(w_b - w_pf, ws_b)):
    plt.plot([mb - sb_, mb + sb_], [i + 1, i + 1], "b-", lw=1, label="PPO Bridge" if i == 0 else None)
plt.xlabel(r"$(v_0-v_{pf})/v_{GT}$"); plt.ylabel("Agent ID"); plt.legend(fontsize=8)
savefig("fig51a_harmonic_offsets");

In [ ]:
estimator_histograms([w_msd, w_pf, w_b], [ws_msd, ws_pf, ws_b],
                     ["MSD", "Particle Filter (2000 particles)", "PPO Agent (10 particles)"], (0.4, 1.2))
savefig("fig51b_harmonic_histograms");

**Figure 51 (p. 124).** Left: estimates from independent observation sets relative to the standard particle filter (red
crosses: its posterior width; black: ground truth; green: MSD; blue: bridged filter). Right: histograms with the reported
accuracy dashed. The MSD underestimates the velocity — the particle appears to travel less because it is caged — while the
bridged filter is unbiased and comparable to the full filter with 200 times fewer particles.